In [1]:
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
import pandas as pd

df = pd.read_csv('merged_with_tweets.csv')
df = df[['congestion_level','timestamp','Temperature','Condition', 'Wind', 'Humidity', 'Pressure', 'Visibility']]
cols_to_clean = ["Temperature", "Humidity", "Pressure", "Visibility", "Wind"]

for col in cols_to_clean:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r"[^\d\.\-]", "", regex=True)
        .replace("", None)
        .astype(float)
    )
df = df.fillna(0)
freq = df["Condition"].value_counts(normalize=True)
df["Condition"] = df["Condition"].map(freq)
df

,congestion_level,timestamp,Temperature,Condition,Wind,Humidity,Pressure,Visibility
0,0,2024-01-06 00:00:00,1.0,0.252246,0.0,54.0,1023.0,16.0
1,0,2024-01-06 00:00:00,1.0,0.252246,0.0,54.0,1023.0,16.0
2,0,2024-01-06 00:00:00,1.0,0.252246,0.0,54.0,1023.0,16.0
3,0,2024-01-06 00:00:00,1.0,0.252246,0.0,54.0,1023.0,16.0
4,0,2024-01-06 00:00:00,1.0,0.252246,0.0,54.0,1023.0,16.0
...,...,...,...,...,...,...,...,...
148941,0,2025-12-11 22:45:00,-2.0,0.092631,20.0,64.0,1010.0,16.0
148942,0,2025-12-11 23:00:00,-2.0,0.092631,20.0,64.0,1010.0,16.0
148943,0,2025-12-11 23:15:00,-2.0,0.092631,20.0,64.0,1010.0,16.0
148944,0,2025-12-11 23:30:00,-2.0,0.092631,20.0,64.0,1010.0,16.0


Frequency encoding for condition

In [2]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
split_idx = int(len(df) * 0.7)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

features = ['Temperature','Condition', 'Wind', 'Humidity', 'Pressure', 'Visibility']
X_train = torch.FloatTensor(train_df[features].values)
y_train = torch.FloatTensor(train_df['congestion_level'].values).unsqueeze(1)
X_test = torch.FloatTensor(test_df[features].values)
y_test = torch.FloatTensor(test_df['congestion_level'].values).unsqueeze(1)


In [3]:
class CongestionModel(nn.Module):
    def __init__(self, input_dim, hidden1=32, hidden2=16, dropout=0.0):
        super(CongestionModel, self).__init__()
        layers = [
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 3)
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = CongestionModel(input_dim=X_train.shape[1])
y_train_int = y_train.view(-1).long()
class_counts = np.bincount(y_train_int.numpy())
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = torch.FloatTensor(class_weights)
class_weights[2] = class_weights[2] * 2
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
epochs = 500

for epoch in range(epochs):
    model.train()
    outputs = model(X_train)
    loss = criterion(outputs, y_train_int)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [10/500], Loss: 48.1993
Epoch [20/500], Loss: 37.9341
Epoch [30/500], Loss: 28.1252
Epoch [40/500], Loss: 19.5692
Epoch [50/500], Loss: 11.6058
Epoch [60/500], Loss: 3.7567
Epoch [70/500], Loss: 2.4776
Epoch [80/500], Loss: 1.4655
Epoch [90/500], Loss: 1.3270
Epoch [100/500], Loss: 1.2795
Epoch [110/500], Loss: 1.2515
Epoch [120/500], Loss: 1.2269
Epoch [130/500], Loss: 1.2088
Epoch [140/500], Loss: 1.1936
Epoch [150/500], Loss: 1.1798
Epoch [160/500], Loss: 1.1668
Epoch [170/500], Loss: 1.1548
Epoch [180/500], Loss: 1.1441
Epoch [190/500], Loss: 1.1344
Epoch [200/500], Loss: 1.1256
Epoch [210/500], Loss: 1.1176
Epoch [220/500], Loss: 1.1105
Epoch [230/500], Loss: 1.1040
Epoch [240/500], Loss: 1.0982
Epoch [250/500], Loss: 1.0929
Epoch [260/500], Loss: 1.0882
Epoch [270/500], Loss: 1.0839
Epoch [280/500], Loss: 1.0799
Epoch [290/500], Loss: 1.0763
Epoch [300/500], Loss: 1.0731
Epoch [310/500], Loss: 1.0700
Epoch [320/500], Loss: 1.0673
Epoch [330/500], Loss: 1.0647
Epoch [340/500

In [4]:
from sklearn.metrics import f1_score, accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
model.eval()
with torch.no_grad():
    logits = model(X_test)
    y_pred = torch.argmax(logits, dim=1)
f1 = f1_score(y_test.numpy(), y_pred.numpy(), average='micro')
precision = precision_score(y_test.numpy(), y_pred.numpy(), average='macro')
recall = recall_score(y_test.numpy(), y_pred.numpy(), average='macro')
cm = classification_report(y_test.numpy(), y_pred.numpy())

print(f1)
print(precision)
print(recall)
print(cm)

0.0506221466296661
0.3171894327089711
0.33101136707984075
              precision    recall  f1-score   support

         0.0       0.84      0.02      0.04     38846
         1.0       0.08      0.07      0.08      4496
         2.0       0.03      0.90      0.06      1342

    accuracy                           0.05     44684
   macro avg       0.32      0.33      0.06     44684
weighted avg       0.74      0.05      0.04     44684



In [ ]:
class CongestionModel(nn.Module):
    def __init__(self, input_dim, hidden1=32, hidden2=16, dropout=0.0):
        super(CongestionModel, self).__init__()
        layers = [
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 3)
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train_int, test_size=0.2, random_state=42)
hidden1_options = [16, 32, 64, 128]
hidden2_options = [8, 16, 32]
dropout_options = [0.0, 0.1, 0.2, 0.3]
lr_options = [1e-4, 5e-4, 1e-3, 5e-3]
batch_size_options = [16, 32, 64]

best_val_loss = float('inf')
best_params = None

for i in range(100):
    h1 = np.random.choice(hidden1_options)
    h2 = np.random.choice(hidden2_options)
    do = np.random.choice(dropout_options)
    lr = np.random.choice(lr_options)
    batch_size = np.random.choice(batch_size_options)

    model = CongestionModel(input_dim=X_train.shape[1], hidden1=h1, hidden2=h2, dropout=do)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    for epoch in range(40):
        model.train()
        permutation = torch.randperm(X_tr.size()[0])
        for j in range(0, X_tr.size()[0], batch_size):
            indices = permutation[j:j+batch_size]
            batch_x, batch_y = X_tr[indices], y_tr[indices]
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        print(f"Epoch [{epoch}], Loss: {loss.item():.4f}")
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = {'hidden1': h1, 'hidden2': h2, 'dropout': do, 'lr': lr, 'batch_size': batch_size}

print("Best Random Search Params:", best_params)
print("Validation Loss:", best_val_loss)

Epoch [0], Loss: 1.3133
Epoch [1], Loss: 0.8969
Epoch [2], Loss: 1.0355
Epoch [3], Loss: 1.2739
Epoch [4], Loss: 1.0740
Epoch [5], Loss: 1.6187
Epoch [6], Loss: 1.0010
Epoch [7], Loss: 1.2266
Epoch [8], Loss: 1.9162
Epoch [9], Loss: 0.9492
Epoch [10], Loss: 1.1293
Epoch [11], Loss: 1.4748
Epoch [12], Loss: 0.9633
Epoch [13], Loss: 0.9423
Epoch [14], Loss: 0.8350
Epoch [15], Loss: 1.2630
Epoch [16], Loss: 0.9381
Epoch [17], Loss: 1.4507
Epoch [18], Loss: 0.9369
Epoch [19], Loss: 1.4918
Epoch [20], Loss: 1.1298
Epoch [21], Loss: 1.0064
Epoch [22], Loss: 1.2480
Epoch [23], Loss: 1.1471
Epoch [24], Loss: 0.9623
Epoch [25], Loss: 0.8890
Epoch [26], Loss: 1.3729
Epoch [27], Loss: 0.9324
Epoch [28], Loss: 1.2573
Epoch [29], Loss: 0.9679
Epoch [30], Loss: 1.3187
Epoch [31], Loss: 0.7956
Epoch [32], Loss: 1.3323
Epoch [33], Loss: 1.2959
Epoch [34], Loss: 0.9870
Epoch [35], Loss: 1.4099
Epoch [36], Loss: 1.5022
Epoch [37], Loss: 1.1373
Epoch [38], Loss: 0.8908
Epoch [39], Loss: 1.0654
Epoch [0],

In [ ]:
import copy
hidden1_grid = [best_params['hidden1']-16, best_params['hidden1'], best_params['hidden1']+16]
hidden2_grid = [best_params['hidden2']-8, best_params['hidden2'], best_params['hidden2']+8]
dropout_grid = [max(0, best_params['dropout']-0.1), best_params['dropout'], best_params['dropout']+0.1]
lr_grid = [best_params['lr']/2, best_params['lr'], best_params['lr']*2]
batch_size = best_params['batch_size']
best_val_loss = float('inf')
best_model = None
best_combination = None
for h1 in hidden1_grid:
    for h2 in hidden2_grid:
        for do in dropout_grid:
            for lr in lr_grid:
                model = CongestionModel(input_dim=X_train.shape[1], hidden1=h1, hidden2=h2, dropout=do)
                optimizer = optim.Adam(model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss(weight=class_weights)
                for epoch in range(40):
                    model.train()
                    permutation = torch.randperm(X_tr.size(0))
                    for i in range(0, X_tr.size(0), batch_size):
                        indices = permutation[i:i+batch_size]
                        batch_x, batch_y = X_tr[indices], y_tr[indices]

                        optimizer.zero_grad()
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                        loss.backward()
                        optimizer.step()
                    print(f"Epoch [{epochs}], Loss: {loss.item():.4f}")
                model.eval()
                with torch.no_grad():
                    val_outputs = model(X_val)
                    val_loss = criterion(val_outputs, y_val).item()

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_model = copy.deepcopy(model)
                    best_combination = {'hidden1': h1, 'hidden2': h2, 'dropout': do, 'lr': lr}

print("Best Grid Search Params:", best_combination)
print("Validation Loss:", best_val_loss)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))


Epoch [10/100], Loss: 1.1610
Epoch [20/100], Loss: 1.1433
Epoch [30/100], Loss: 1.1463
Epoch [40/100], Loss: 1.1389
Epoch [10/100], Loss: 1.1601
Epoch [20/100], Loss: 1.1425
Epoch [30/100], Loss: 1.0262
Epoch [40/100], Loss: 1.1638
Epoch [10/100], Loss: 1.0420
Epoch [20/100], Loss: 1.1377
Epoch [30/100], Loss: 1.0745
Epoch [40/100], Loss: 1.1731
Epoch [10/100], Loss: 1.1724
Epoch [20/100], Loss: 1.1699
Epoch [30/100], Loss: 1.0599
Epoch [40/100], Loss: 1.0411
Epoch [10/100], Loss: 0.9928
Epoch [20/100], Loss: 1.0419
Epoch [30/100], Loss: 1.1736
Epoch [40/100], Loss: 1.0418
Epoch [10/100], Loss: 1.0596
Epoch [20/100], Loss: 1.1606
Epoch [30/100], Loss: 1.1748
Epoch [40/100], Loss: 1.1703
Epoch [10/100], Loss: 1.1740
Epoch [20/100], Loss: 1.1465
Epoch [30/100], Loss: 1.0156
Epoch [40/100], Loss: 1.0854
Epoch [10/100], Loss: 1.0224
Epoch [20/100], Loss: 1.0750
Epoch [30/100], Loss: 1.1643
Epoch [40/100], Loss: 1.1664
Epoch [10/100], Loss: 1.1835
Epoch [20/100], Loss: 1.1830
Epoch [30/100]

In [ ]:
final_hidden1 = best_combination['hidden1']
final_hidden2 = best_combination['hidden2']
final_dropout = best_combination['dropout']
final_lr = best_combination['lr']
batch_size = best_params['batch_size']
final_model = CongestionModel(
    input_dim=X_train.shape[1],
    hidden1=final_hidden1,
    hidden2=final_hidden2,
    dropout=final_dropout
)

optimizer = optim.Adam(final_model.parameters(), lr=final_lr)
criterion = nn.CrossEntropyLoss(weight=class_weights)
epochs = 200
for epoch in range(epochs):
    final_model.train()
    permutation = torch.randperm(X_train.size(0))

    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train[indices], y_train_int[indices]

        optimizer.zero_grad()
        outputs = final_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [20/200], Loss: 1.1400
Epoch [40/200], Loss: 0.8132
Epoch [60/200], Loss: 1.1351
Epoch [80/200], Loss: 0.8584
Epoch [100/200], Loss: 0.6892
Epoch [120/200], Loss: 1.1268
Epoch [140/200], Loss: 1.2737
Epoch [160/200], Loss: 1.1918
Epoch [180/200], Loss: 1.0317
Epoch [200/200], Loss: 1.1378


In [ ]:
final_model.eval()
with torch.no_grad():
    logits = final_model(X_test)
    y_pred = torch.argmax(logits, dim=1)
f1 = f1_score(y_test.numpy(), y_pred.numpy(), average='micro')
precision = precision_score(y_test.numpy(), y_pred.numpy(), average='macro')
recall = recall_score(y_test.numpy(), y_pred.numpy(), average='macro')
cm = classification_report(y_test.numpy(), y_pred.numpy())

print(f1)
print(precision)
print(recall)
print(cm)

0.2893653209202399
0.34805347934350755
0.3580268909280049
              precision    recall  f1-score   support

         0.0       0.89      0.28      0.43     38846
         1.0       0.13      0.29      0.18      4496
         2.0       0.03      0.50      0.06      1342

    accuracy                           0.29     44684
   macro avg       0.35      0.36      0.22     44684
weighted avg       0.78      0.29      0.39     44684



In [5]:
class CongestionEncoder(nn.Module):
    def __init__(self, trained_model):
        super(CongestionEncoder, self).__init__()
        self.encoder = nn.Sequential(
            trained_model.net[0],
            trained_model.net[1],
            trained_model.net[2],
            trained_model.net[3],
            trained_model.net[4],
            trained_model.net[5]
        )

    def forward(self, x):
        return self.encoder(x)
encoder = CongestionEncoder(model)
encoder.eval()
sample_batch = X_train[:5]
with torch.no_grad():
    embeddings = encoder(sample_batch)
print("Embeddings shape:", embeddings.shape)
print("Embeddings:")
print(embeddings)

Embeddings shape: torch.Size([5, 16])
Embeddings:
tensor([[ 23.1306,   0.0000,   0.0000,  82.9915, 157.7644, 147.0542,   0.0000,
         114.3495,   0.0000,   0.0000,   0.0000, 142.5081,   0.0000, 270.7770,
           0.0000, 114.8432],
        [ 23.1306,   0.0000,   0.0000,  82.9915, 157.7644, 147.0542,   0.0000,
         114.3495,   0.0000,   0.0000,   0.0000, 142.5081,   0.0000, 270.7770,
           0.0000, 114.8432],
        [ 23.1306,   0.0000,   0.0000,  82.9915, 157.7644, 147.0542,   0.0000,
         114.3495,   0.0000,   0.0000,   0.0000, 142.5081,   0.0000, 270.7770,
           0.0000, 114.8432],
        [ 23.1306,   0.0000,   0.0000,  82.9915, 157.7644, 147.0542,   0.0000,
         114.3495,   0.0000,   0.0000,   0.0000, 142.5081,   0.0000, 270.7770,
           0.0000, 114.8432],
        [ 23.1306,   0.0000,   0.0000,  82.9915, 157.7644, 147.0542,   0.0000,
         114.3495,   0.0000,   0.0000,   0.0000, 142.5081,   0.0000, 270.7770,
           0.0000, 114.8432]])


In [6]:
model.eval()
input_size = X_train.shape[-1]
example_input = torch.randn(1, input_size)
scripted_model = torch.jit.trace(encoder, example_input)

torch.jit.save(scripted_model, "weather_weights.pt")

In [ ]:
torch.save(model, "weather_weights.pth")

In [ ]:
from google.colab import files

files.download('weather_weights.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>